In [ ]:
import pandas as pd
import torch

# Install required libraries

In [ ]:
!pip install transformers datasets accelerate wandb

## Wandb setup

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 21f2000763 (21f2000763-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
wandb.init(
    project="21f2000763-t22026",
    name="deberta-v3-finetune-run3",
    tags = ["deberta", "transformer"]
)


## Load the Tokenizer

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
test_sentence = 'What is emissivity?'
inputs = tokenizer(test_sentence)
print(inputs)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

{'input_ids': [1, 458, 269, 865, 23566, 21555, 302, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
import transformers
print(transformers.__version__)

5.12.0


## Create Dataset


In [ ]:
train_df = pd.read_csv("/content/train.csv")

In [ ]:
from datasets import Dataset
dataset = Dataset.from_pandas(train_df)
print(dataset)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


In [ ]:
# Split the data into training and validation data.
split_dataset = dataset.train_test_split(test_size = 0.2)
split_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
        num_rows: 400
    })
})

In [ ]:
split_dataset["train"][0]["prompt"]

'Select the most accurate option: What is the formalism that angular momentum is associated with in rotational invariance? from the following choices.'

In [ ]:
split_dataset["train"]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 1600
})

## Tokenization

In [ ]:
## Tokenization
def preprocess(data):
  output = {
      "input_ids": [],
      "attention_mask": [],
      "labels": []
  }

  label_map = {"A": 0, "B": 1, "C":2, "D":3, "E": 4}
  prompts = []
  options = []
  answer = []
  for i in range(len(data["prompt"])):
    prompts += [data["prompt"][i]]*5
    options += [data[x][i] for x in label_map.keys()]
    answer.append(label_map[data["answer"][i]])

  tokenized_input = tokenizer(prompts, options, truncation=True, max_length=256)

  for i in range(0,len(tokenized_input["input_ids"]),5):
    tokenized_return = {
      "input_ids": [],
      "attention_mask": [],
    }
    tokenized_return["input_ids"] += tokenized_input["input_ids"][i:i+5]
    tokenized_return["attention_mask"] += tokenized_input["attention_mask"][i:i+5]
    tokenized_return["labels"] = answer[i//5]


    output["input_ids"].append(tokenized_return["input_ids"])
    output["attention_mask"].append(tokenized_return["attention_mask"])
    output["labels"].append(tokenized_return["labels"])

  print(output["input_ids"][0])

  return output


In [ ]:
tokenized_dataset = split_dataset.map(preprocess, batched = True)
print(tokenized_dataset)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

[[1, 5200, 262, 370, 3027, 1209, 294, 458, 269, 262, 78949, 272, 23032, 7365, 269, 1635, 275, 267, 32865, 267, 78116, 302, 292, 262, 776, 2811, 260, 2, 40038, 7365, 269, 262, 376, 271, 9930, 585, 36681, 1541, 1635, 275, 32865, 267, 78116, 260, 2], [1, 5200, 262, 370, 3027, 1209, 294, 458, 269, 262, 78949, 272, 23032, 7365, 269, 1635, 275, 267, 32865, 267, 78116, 302, 292, 262, 776, 2811, 260, 2, 40038, 7365, 269, 262, 404, 271, 9930, 585, 36681, 1541, 1635, 275, 32865, 267, 78116, 260, 2], [1, 5200, 262, 370, 3027, 1209, 294, 458, 269, 262, 78949, 272, 23032, 7365, 269, 1635, 275, 267, 32865, 267, 78116, 302, 292, 262, 776, 2811, 260, 2, 40038, 7365, 269, 262, 456, 271, 9930, 585, 36681, 1541, 1635, 275, 32865, 267, 78116, 260, 2], [1, 5200, 262, 370, 3027, 1209, 294, 458, 269, 262, 78949, 272, 23032, 7365, 269, 1635, 275, 267, 32865, 267, 78116, 302, 292, 262, 776, 2811, 260, 2, 40038, 7365, 269, 262, 392, 271, 9930, 585, 36681, 1541, 1635, 275, 32865, 267, 78116, 260, 2], [1, 5200, 2

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

[[1, 577, 386, 120176, 582, 1303, 281, 343, 267, 475, 271, 10074, 754, 302, 2, 574, 2], [1, 577, 386, 120176, 582, 1303, 281, 343, 267, 475, 271, 10074, 754, 302, 2, 2686, 2], [1, 577, 386, 120176, 582, 1303, 281, 343, 267, 475, 271, 10074, 754, 302, 2, 936, 2], [1, 577, 386, 120176, 582, 1303, 281, 343, 267, 475, 271, 10074, 754, 302, 2, 456, 2], [1, 577, 386, 120176, 582, 1303, 281, 343, 267, 475, 271, 10074, 754, 302, 2, 1824, 2]]
DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 400
    })
})


In [ ]:
from dataclasses import dataclass
from transformers import DataCollatorWithPadding

In [ ]:
@dataclass
class DataCollatorForMultipleChoice:
  tokenizer: any

  def __call__(self,features):
    flat_input = {
        "input_ids": [],
        "attention_mask": [],
    }
    labels = []

    for item in features:
      flat_input["input_ids"] += item["input_ids"]
      flat_input["attention_mask"] += item["attention_mask"]
      labels.append(item["labels"])

    batch = self.tokenizer.pad(
        flat_input,
        return_tensors = "pt"
    )

    batch["input_ids"] = batch["input_ids"].view(len(features),5,-1)
    batch["attention_mask"] = batch["attention_mask"].view(len(features), 5, -1)
    batch["labels"] = torch.tensor(labels)

    return batch



In [ ]:
# Instantiate the collator
data_collator = DataCollatorForMultipleChoice(tokenizer = tokenizer)

In [ ]:
# Import the Model
from transformers import AutoModelForMultipleChoice
model = AutoModelForMultipleChoice.from_pretrained("microsoft/deberta-v3-base", torch_dtype=torch.float32)
print(next(model.parameters()).dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias 

torch.float32


In [ ]:
from transformers import TrainingArguments, Trainer
# Set up training arguments

training_args = TrainingArguments(
    output_dir = "./results_v2",
    save_strategy = "epoch",
    learning_rate = 1e-5,
    per_device_train_batch_size = 4,
    num_train_epochs = 3,
    weight_decay = 0.01,
    fp16=False,
    max_grad_norm=1.0,
    report_to="wandb",
    logging_steps=50,
    run_name = "deberta-v3-fintune-run3"

)

In [ ]:
# Initialize the Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["test"],
    data_collator = data_collator,
)

In [ ]:
trainer.train()


Step,Training Loss
50,1.612996
100,1.609814
150,1.520018
200,1.463537
250,1.401820
300,1.213727
350,1.068483
400,0.945659
450,0.779545
500,0.710149


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.7699409484863281, metrics={'train_runtime': 554.5515, 'train_samples_per_second': 8.656, 'train_steps_per_second': 2.164, 'total_flos': 1073091604557120.0, 'train_loss': 0.7699409484863281, 'epoch': 3.0})

In [ ]:
has_nan = False
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print("NaN found:", name)
        has_nan = True
        break

print("Model dtype:", next(model.parameters()).dtype)

if not has_nan:
    print("Model is OK after training.")

Model dtype: torch.float32
Model is OK after training.


## Predict on test data.

In [ ]:
test_df = pd.read_csv("/content/test.csv")

In [ ]:
# add an answer column with a placeholder option.
test_df["answer"] = "A"

In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      500 non-null    int64 
 1   prompt  500 non-null    object
 2   A       500 non-null    object
 3   B       500 non-null    object
 4   C       500 non-null    object
 5   D       500 non-null    object
 6   E       500 non-null    object
 7   answer  500 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.4+ KB


In [ ]:
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
print(test_dataset)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 500
})


In [ ]:
tokenized_test_dataset = test_dataset.map(preprocess, batched = True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[[1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 262, 454, 843, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 266, 1026, 843, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 266, 467, 5465, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 26

In [ ]:
test_predictions = trainer.predict(tokenized_test_dataset)

for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print("NaN after predict:", name)
        break

wandb.finish()

test/loss,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇████
train/grad_norm,▁▁▂▂▂▂▃▃▃▁▃▃▁█▄▃▁▂▁▁▃▁▂▂
train/learning_rate,██▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁
train/loss,███▇▇▆▅▅▄▃▃▃▃▂▂▂▂▁▁▂▁▁▁▁
test/loss,4.79891
test/runtime,19.2903


In [ ]:
print(test_predictions)

PredictionOutput(predictions=array([[ 5.6061125 ,  1.4027077 ,  1.0627916 ,  1.7488447 ,  3.0005324 ],
       [ 0.38032687,  4.901065  ,  0.59675586,  0.4249407 ,  0.40677088],
       [-0.16938537,  7.4600773 , -0.20027317,  0.16231674,  0.31958055],
       ...,
       [ 0.39548713,  5.7549634 ,  0.5309692 ,  0.80720145,  0.32764506],
       [ 0.18660328,  1.8671602 ,  0.33793575,  0.07478617,  4.9588885 ],
       [ 0.08594281, -0.03807216,  7.7513747 ,  0.25366473, -0.13373606]],
      dtype=float32), label_ids=array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
print(model.dtype)

torch.float32


In [ ]:
next(model.parameters()).dtype

torch.float32

In [ ]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


In [ ]:
import numpy as np
logits = test_predictions.predictions
print(logits)

[[ 5.6061125   1.4027077   1.0627916   1.7488447   3.0005324 ]
 [ 0.38032687  4.901065    0.59675586  0.4249407   0.40677088]
 [-0.16938537  7.4600773  -0.20027317  0.16231674  0.31958055]
 ...
 [ 0.39548713  5.7549634   0.5309692   0.80720145  0.32764506]
 [ 0.18660328  1.8671602   0.33793575  0.07478617  4.9588885 ]
 [ 0.08594281 -0.03807216  7.7513747   0.25366473 -0.13373606]]


In [ ]:
top_3_indices = np.argsort(logits, axis=1)[:,-1:-4:-1]

In [ ]:
index_to_letter = {0:"A", 1: "B", 2: "C", 3: "D", 4: "E"}

In [ ]:
predicted_strings = []
for row in top_3_indices:
  letters = [index_to_letter[idx] for idx in row]
  predicted_strings.append(" ".join(letters))

In [ ]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predicted_strings
})

In [ ]:
submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())

   id prediction
0   1      A E D
1   2      B C D
2   3      B E D
3   4      E C A
4   5      C D A


## Save the model

In [ ]:
trainer.save_model("./my_deberta_model")
tokenizer.save_pretrained("./my_deberta_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_deberta_model/tokenizer_config.json',
 './my_deberta_model/tokenizer.json')

In [ ]:
import shutil
shutil.make_archive("my_deberta_model", 'zip', "./my_deberta_model")

'/content/my_deberta_model.zip'